# MambaVision-T — Indiana Chest X-ray | EXPERIMENT: Stage 4 → SpatialMamba (Kaggle)
### Stage 4 replaced with 4-directional SpatialMambaStage | Identical structure to baseline

**Dataset:** Indiana University Chest X-ray (Kaggle) — same split as baseline

**MambaVision:** Upload your **modified** `MambaVision/` folder (with `SpatialMambaStage`
in `mamba_vision.py`) as a separate Kaggle dataset, e.g. `mambavision-spatial`.

**What changed vs baseline:**
- Stage 4 (`i=3`): `MambaVisionLayer` → `SpatialMambaStage`
- 4-directional SSM (L→R, R→L, T→B, B→T) + learnable `Linear(4C→C)` fusion
- Stages 1–3 and all training config **identical to baseline**

| Cell | Content | Note |
|---|---|---|
| 1 | Imports | |
| 2 | PyTorch 2.6 patch | |
| 3 | Cache clear + MambaVision path (modified) | |
| 4 | Stage verification | confirms SpatialMambaStage |
| 5 | Dataset class | same as baseline |
| 6 | DataLoaders | same split seed |
| 7 | Load model — Stage 4 = SpatialMambaStage | |
| 8 | FLOPs & params | will differ from baseline |
| 9 | Loss / Optimiser / Scheduler | identical to baseline |
| 10 | Train + Validate functions | identical to baseline |
| 11 | Training loop | GPU peak tracked here |
| 12 | Load best model | |
| 13 | Inference time | after training |
| 14 | GPU memory | after training |
| 15 | Final evaluation | all 13 metrics |
| 16 | Training curves | |
| 17 | Per-class visualisation | |
| 18 | Benchmark summary | |
| 19 | Save JSON + TXT | |

## Cell 1 — Imports & Environment

In [1]:
import os, sys, time, json, warnings, math, random, shutil
from datetime import datetime
from pathlib import Path
from contextlib import nullcontext
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    matthews_corrcoef, hamming_loss
)
from sklearn.model_selection import train_test_split

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ── Install & import FLOPs library ───────────────────────────────────
FLOPS_AVAILABLE = False
THOP_AVAILABLE  = False
import subprocess as _sp

def _pip_install(pkg):
    r = _sp.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], capture_output=True)
    return r.returncode == 0

# 1) Try fvcore (exact FLOPs)
try:
    from fvcore.nn import FlopCounterMode
    FLOPS_AVAILABLE = True
    print("✓ fvcore already available — exact FLOPs")
except ImportError:
    if _pip_install('fvcore'):
        try:
            from fvcore.nn import FlopCounterMode
            FLOPS_AVAILABLE = True
            print("✓ fvcore installed — exact FLOPs")
        except ImportError:
            pass

# 2) Fall back to thop (approx FLOPs)
if not FLOPS_AVAILABLE:
    try:
        from thop import profile as thop_profile
        THOP_AVAILABLE = True
        print("✓ thop already available — approx FLOPs")
    except ImportError:
        if _pip_install('thop'):
            try:
                from thop import profile as thop_profile
                THOP_AVAILABLE = True
                print("✓ thop installed — approx FLOPs")
            except ImportError:
                pass

if not FLOPS_AVAILABLE and not THOP_AVAILABLE:
    print("⚠ no FLOPs library available — will use paper estimate")

# ── Environment ───────────────────────────────────────────────────────
print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA ver : {torch.version.cuda}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    TOTAL_VRAM_MB = props.total_memory / 1024**2
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {TOTAL_VRAM_MB/1024:.2f} GB  ({TOTAL_VRAM_MB:.0f} MB)")
else:
    TOTAL_VRAM_MB = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {device}")

# ── GPU SM detection (for P100 / sm_60 compatibility) ────────────────
GPU_SM = 0
if torch.cuda.is_available():
    GPU_SM = props.major * 10 + props.minor
    print(f"SM       : sm_{GPU_SM}  (need >= sm_70 for mamba-ssm CUDA kernels)")

# mamba-ssm requires sm_70+ (V100, T4, A100 …)
# P100 = sm_60 → CUDA kernels crash → use pure-PyTorch fallback
USE_MAMBA_SSM_CUDA = (GPU_SM >= 70)
print(f"\nUse mamba-ssm CUDA kernels : {USE_MAMBA_SSM_CUDA}")
if not USE_MAMBA_SSM_CUDA:
    print("  ↳ GPU is P100/older — will use pure-PyTorch SSM fallback in Cell 3")

✓ thop installed — approx FLOPs

PyTorch  : 2.10.0+cu128
CUDA ver : 12.8
CUDA OK  : True
GPU      : Tesla T4
VRAM     : 14.56 GB  (14913 MB)
Device   : cuda
SM       : sm_75  (need >= sm_70 for mamba-ssm CUDA kernels)

Use mamba-ssm CUDA kernels : True


## Cell 2 — PyTorch 2.6 Patch

In [2]:
import argparse
try:
    import torch.serialization
    torch.serialization.add_safe_globals([argparse.Namespace])
    print("✓ argparse.Namespace allowlisted (PyTorch 2.6 fix)")
except AttributeError:
    print("✓ PyTorch < 2.6 — no patch needed")

_orig_load = torch.load
def _safe_load(f, map_location=None, pickle_module=None, weights_only=False, **kw):
    return _orig_load(f, map_location=map_location, weights_only=False, **kw)
torch.load = _safe_load
print("✓ torch.load monkey-patched → weights_only=False")

✓ argparse.Namespace allowlisted (PyTorch 2.6 fix)
✓ torch.load monkey-patched → weights_only=False


## Cell 3 — Cache Clear & Modified MambaVision Path
> Upload the **modified** MambaVision folder (containing `SpatialMambaStage`) as a
> separate Kaggle dataset, e.g. `mambavision-spatial`.

In [3]:
import shutil
import importlib

# ── Locate the modified MambaVision checkout/dataset ─────────────────
CANDIDATE_ROOTS = [
    Path('/kaggle/input/datasets/walliullah527/mambasp/MambaVision'),
    Path.cwd() / 'MambaVision',
    Path('/Users/qaiserfarooq/Documents/experiment/MambaVision'),
]
MAMBA_VISION_ROOT_PATH = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
if MAMBA_VISION_ROOT_PATH is None:
    raise FileNotFoundError(
        'Could not find a MambaVision root. Checked: ' + ', '.join(str(p) for p in CANDIDATE_ROOTS)
    )
MAMBA_VISION_ROOT = str(MAMBA_VISION_ROOT_PATH)
print(f"\u2713 MambaVision root: {MAMBA_VISION_ROOT}")

# Clear bytecode cache
pyc_n = cache_n = 0
for root, dirs, files in os.walk(MAMBA_VISION_ROOT):
    for f in files:
        if f.endswith('.pyc'):
            os.remove(os.path.join(root, f)); pyc_n += 1
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d)); cache_n += 1
print(f"\u2713 Cache cleared \u2014 {pyc_n} .pyc, {cache_n} __pycache__ removed")

stale = [k for k in sys.modules if 'mambavision' in k or 'mamba_vision' in k or 'spatial_mamba' in k]
for k in stale: del sys.modules[k]
print(f"\u2713 {len(stale)} stale modules removed from sys.modules")

for p in [MAMBA_VISION_ROOT, os.path.dirname(MAMBA_VISION_ROOT)]:
    if p not in sys.path: sys.path.insert(0, p)
    print(f"  sys.path \u2190 {p}")

model_file = os.path.join(MAMBA_VISION_ROOT,'mambavision','models','mamba_vision.py')
if not os.path.exists(model_file):
    raise FileNotFoundError(f"mamba_vision.py not found at {model_file}")
with open(model_file) as f: src = f.read()
if 'SpatialMambaLayer' not in src:
    raise RuntimeError('SpatialMambaLayer not found \u2014 upload the modified mamba_vision.py')
print('  \u2713 SpatialMambaLayer in mamba_vision.py')

if USE_MAMBA_SSM_CUDA:
    print("\nInstalling mamba-ssm...")
    os.system(f'"{sys.executable}" -m pip install mamba-ssm --no-build-isolation -q')
    try:
        from mamba_ssm import Mamba
        print("\u2713 mamba-ssm CUDA imported successfully")
    except ImportError as e:
        print(f"\u2717 mamba-ssm import failed: {e}")
        USE_MAMBA_SSM_CUDA = False

if not USE_MAMBA_SSM_CUDA:
    print("\nInjecting pure-PyTorch Mamba drop-in (P100-safe)...")

    def _selective_scan_fn(x, *args, **kwargs):
        return x

    class _MambaPurePyTorch(nn.Module):
        # FIX (Bug 3): removed inner self.norm.
        # SpatialMambaBlock applies norm1/norm2 externally; double-normalising
        # suppressed gradients and prevented loss from decreasing.
        def __init__(self, d_model, d_state=16, d_conv=4,
                     expand=2, dt_rank='auto', **kwargs):
            super().__init__()
            self.d_model = d_model
            inner = int(d_model * expand)
            self.in_proj  = nn.Linear(d_model, inner * 2, bias=False)
            self.gru      = nn.GRU(inner, inner, batch_first=True,
                                   bidirectional=False)
            self.out_proj = nn.Linear(inner, d_model, bias=False)
            # No self.norm here: normalisation is owned by SpatialMambaBlock.

        def forward(self, x):
            B, L, D = x.shape
            xz = self.in_proj(x)
            xi, z = xz.chunk(2, dim=-1)
            xi, _ = self.gru(xi)
            xi = xi * torch.sigmoid(z)
            # FIX: plain residual without LayerNorm
            return x + self.out_proj(xi)

    import types
    mamba_ssm_mock = types.ModuleType('mamba_ssm')
    mamba_ssm_mock.__path__ = []
    mamba_ssm_mock.Mamba = _MambaPurePyTorch
    sys.modules['mamba_ssm'] = mamba_ssm_mock

    ops_mock = types.ModuleType('mamba_ssm.ops')
    ops_mock.__path__ = []
    sys.modules['mamba_ssm.ops'] = ops_mock

    selective_mock = types.ModuleType('mamba_ssm.ops.selective_scan_interface')
    selective_mock.selective_scan_fn = _selective_scan_fn
    sys.modules['mamba_ssm.ops.selective_scan_interface'] = selective_mock

    if 'causal_conv1d' not in sys.modules:
        causal_mock = types.ModuleType('causal_conv1d')
        causal_mock.__path__ = []
        sys.modules['causal_conv1d'] = causal_mock
        sys.modules['causal_conv1d.causal_conv1d_cuda'] = types.ModuleType('causal_conv1d.causal_conv1d_cuda')

    print("  \u2713 _MambaPurePyTorch injected as mamba_ssm.Mamba (fixed: no double-norm)")
    print("  \u2713 selective_scan_fn stub injected")
    print("  \u2713 No CUDA kernels needed \u2014 P100 / CPU compatible")

# ── FIX (Bug 6): removed inline SpatialMambaBlock/SpatialMambaLayer definitions.
# Those redefinitions shadowed spatial_mamba.py (broken DropPath, wrong FFN,
# wrong residual order). We now reload the .py file directly and use its classes.
import mambavision.models.spatial_mamba as _sm
importlib.reload(_sm)
# The mamba_ssm mock is already in sys.modules, so the reloaded _sm automatically
# uses _MambaPurePyTorch on P100 paths without any extra patching.

importlib.reload(importlib.import_module('mambavision.models.mamba_vision'))
from mambavision.models.mamba_vision import mamba_vision_T
print(f"\n\u2713 mamba_vision_T imported from: {importlib.import_module('mambavision.models.mamba_vision').__file__}")
print(f"  Mamba backend: {'mamba-ssm CUDA' if USE_MAMBA_SSM_CUDA else 'pure-PyTorch GRU (P100-safe)'}")
print(f"  SpatialMambaBlock: loaded from spatial_mamba.py (all bugs fixed)")


✓ MambaVision root: /kaggle/input/datasets/walliullah527/mambasp/MambaVision
✓ Cache cleared — 0 .pyc, 0 __pycache__ removed
✓ 0 stale modules removed from sys.modules
  sys.path ← /kaggle/input/datasets/walliullah527/mambasp/MambaVision
  sys.path ← /kaggle/input/datasets/walliullah527/mambasp
  ✓ SpatialMambaLayer in mamba_vision.py

Installing mamba-ssm...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 4.7 MB/s eta 0:00:00
✓ mamba-ssm CUDA imported successfully

✓ mamba_vision_T imported from: /kaggle/input/datasets/walliullah527/mambasp/MambaVision/mambavision/models/mamba_vision.py
  Mamba backend: mamba-ssm CUDA
  SpatialMambaBlock: loaded from spatial_mamba.py (all bugs fixed)


## Cell 4 — Stage Architecture Verification
> Must pass before continuing.

In [4]:
_chk = mamba_vision_T(pretrained=False, num_classes=14)
print("Stage architecture:")
for i in range(4):
    print(f"  Stage {i+1} (i={i}): {type(_chk.levels[i]).__name__}")

s4_type = type(_chk.levels[3]).__name__
s4_blk  = type(_chk.levels[3].blocks[0]).__name__
blk     = _chk.levels[3].blocks[0]

checks = {
    'SpatialMambaLayer at i=3': s4_type == 'SpatialMambaLayer',
    'SpatialMambaBlock inside':  s4_blk  == 'SpatialMambaBlock',
    'mamba module present':      hasattr(blk, 'mamba'),
    'FFN present':               hasattr(blk, 'ffn'),
    'norm1 present':             hasattr(blk, 'norm1'),
    'norm2 present':             hasattr(blk, 'norm2'),
}
all_ok = True
for desc, ok in checks.items():
    print(f"  {'✓' if ok else '✗'}  {desc}")
    if not ok: all_ok = False

s4_params = sum(p.numel() for p in _chk.levels[3].parameters())
print(f"\n  Stage 4 params: {s4_params:,}  (SpatialMambaLayer)")
print(f"\n{'✓ ALL CHECKS PASSED' if all_ok else '✗ CHECKS FAILED'}")
if not all_ok:
    raise RuntimeError("Stage 4 verification failed.")
del _chk; print("  (throwaway model deleted)")

Stage architecture:
  Stage 1 (i=0): MambaVisionLayer
  Stage 2 (i=1): MambaVisionLayer
  Stage 3 (i=2): MambaVisionLayer
  Stage 4 (i=3): SpatialMambaLayer
  ✓  SpatialMambaLayer at i=3
  ✓  SpatialMambaBlock inside
  ✓  mamba module present
  ✓  FFN present
  ✓  norm1 present
  ✓  norm2 present

  Stage 4 params: 9,349,760  (SpatialMambaLayer)

✓ ALL CHECKS PASSED
  (throwaway model deleted)


## Cell 5 — Indiana Dataset Class

In [5]:
class IndianaChestXrayDataset(Dataset):
    """
    Indiana University Chest X-ray Dataset (Kaggle).
    Multi-label, 14 disease classes — same label set as NIH ChestX-ray14
    so all metric code stays identical.

    Kaggle dataset: chest-xrays-indiana-university
    Structure:
        /kaggle/input/chest-xrays-indiana-university/
            indiana_reports.csv    ← findings / impression text (not used for labels)
            indiana_projections.csv ← maps uid to image filename + projection
            images/images_normalized/  ← all PNG images

    Labels are parsed from the 'Problems' column in indiana_projections.csv
    which contains pipe-separated disease tags matching NIH class names.
    """

    ALL_CLASSES = [
        'Atelectasis','Consolidation','Infiltration','Pneumothorax',
        'Edema','Emphysema','Fibrosis','Effusion','Pneumonia',
        'Pleural_Thickening','Cardiomegaly','Nodule','Mass','Hernia'
    ]

    # Indiana dataset uses slightly different tag spellings — map them
    TAG_MAP = {
        'pleural effusion':      'Effusion',
        'effusion':              'Effusion',
        'pneumothorax':          'Pneumothorax',
        'cardiomegaly':          'Cardiomegaly',
        'edema':                 'Edema',
        'pulmonary edema':       'Edema',
        'consolidation':         'Consolidation',
        'infiltrate':            'Infiltration',
        'infiltration':          'Infiltration',
        'pneumonia':             'Pneumonia',
        'atelectasis':           'Atelectasis',
        'emphysema':             'Emphysema',
        'fibrosis':              'Fibrosis',
        'pleural thickening':    'Pleural_Thickening',
        'nodule':                'Nodule',
        'mass':                  'Mass',
        'hernia':                'Hernia',
        'hiatal hernia':         'Hernia',
    }

    def __init__(self, image_dir, records, transform=None, target_size=224):
        """
        records: list of dicts with keys 'filename' and 'labels' (np.array len 14)
        """
        self.image_dir   = image_dir
        self.records     = records
        self.transform   = transform
        self.target_size = target_size
        print(f"  Dataset: {len(self.records):,} images")

    def __len__(self): return len(self.records)

    def _resize_pad(self, img, sz):
        h, w  = img.shape[:2]
        scale = min(sz/h, sz/w)
        nh, nw = int(h*scale), int(w*scale)
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        canvas = np.zeros((sz, sz, 3), dtype=img.dtype)
        y0 = (sz-nh)//2; x0 = (sz-nw)//2
        canvas[y0:y0+nh, x0:x0+nw] = img
        return canvas

    def __getitem__(self, idx):
        rec  = self.records[idx]
        path = os.path.join(self.image_dir, rec['filename'])
        img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # Try with .png extension if not found
            path = path if path.endswith('.png') else path + '.png'
            img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((224, 224), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = self._resize_pad(img, self.target_size)
        img = self.transform(img) if self.transform else               torch.from_numpy(img).permute(2,0,1).float()/255.
        return img, torch.from_numpy(rec['labels'])


def build_indiana_records(proj_csv_path, image_dir):
    """
    Parse indiana_projections.csv + indiana_reports.csv → list of records.

    The Indiana dataset has labels ONLY in indiana_reports.csv (findings/impression
    free-text), NOT in indiana_projections.csv which only has uid/filename/projection.
    Labels are extracted by keyword-matching against the findings+impression text.

    Each record: {'filename': str, 'labels': np.float32 array len 14}
    Only keeps frontal (PA or AP) projections.
    """
    proj = pd.read_csv(proj_csv_path)
    print(f"  Projections CSV: {len(proj):,} rows, columns: {proj.columns.tolist()}")

    # ── Load reports CSV (lives next to projections CSV) ──────────────
    reports_csv = os.path.join(os.path.dirname(proj_csv_path), 'indiana_reports.csv')
    if not os.path.exists(reports_csv):
        raise FileNotFoundError(
            f"indiana_reports.csv not found at {reports_csv}\n"
            "This file is required for label extraction."
        )
    reports = pd.read_csv(reports_csv)
    print(f"  Reports CSV    : {len(reports):,} rows, columns: {reports.columns.tolist()}")

    # ── Identify columns ──────────────────────────────────────────────
    # projections: uid | filename | projection
    fn_col  = next((c for c in proj.columns if 'filename' in c.lower()), proj.columns[1])
    prj_col = next((c for c in proj.columns if 'projection' in c.lower()), None)
    uid_col_proj = next((c for c in proj.columns if 'uid' in c.lower()), proj.columns[0])

    # reports: uid | findings | impression  (column names vary slightly)
    uid_col_rep  = next((c for c in reports.columns if 'uid' in c.lower()), reports.columns[0])
    find_col = next((c for c in reports.columns if 'finding' in c.lower()), None)
    impr_col = next((c for c in reports.columns if 'impression' in c.lower()), None)
    print(f"  Proj  → uid='{uid_col_proj}'  filename='{fn_col}'  projection='{prj_col}'")
    print(f"  Rep   → uid='{uid_col_rep}'  findings='{find_col}'  impression='{impr_col}'")

    # ── Build uid → free-text map ─────────────────────────────────────
    uid_to_text = {}
    for _, row in reports.iterrows():
        uid = str(row[uid_col_rep]).strip()
        parts = []
        if find_col and pd.notna(row[find_col]):
            parts.append(str(row[find_col]))
        if impr_col and pd.notna(row[impr_col]):
            parts.append(str(row[impr_col]))
        uid_to_text[uid] = ' '.join(parts).lower()

    print(f"  uid→text entries: {len(uid_to_text):,}")

    # ── Keyword map: term → class  (generous, substring-based) ───────
    tag_map = IndianaChestXrayDataset.TAG_MAP
    all_cls = IndianaChestXrayDataset.ALL_CLASSES

    # Additional multi-word keyword → class map for free-text matching
    KEYWORD_MAP = {
        'atelectasis':        'Atelectasis',
        'atelectatic':        'Atelectasis',
        'consolidat':         'Consolidation',
        'infiltrat':          'Infiltration',
        'pneumothorax':       'Pneumothorax',
        'edema':              'Edema',
        'emphysema':          'Emphysema',
        'fibrosis':           'Fibrosis',
        'fibrotic':           'Fibrosis',
        'pleural effusion':   'Effusion',
        'effusion':           'Effusion',
        'pneumonia':          'Pneumonia',
        'pleural thickening': 'Pleural_Thickening',
        'pleural thick':      'Pleural_Thickening',
        'cardiomegaly':       'Cardiomegaly',
        'cardiac enlargement':'Cardiomegaly',
        'enlarged heart':     'Cardiomegaly',
        'nodule':             'Nodule',
        'nodular':            'Nodule',
        'mass':               'Mass',
        'hernia':             'Hernia',
    }

    # ── Build records ─────────────────────────────────────────────────
    records = []
    for _, row in proj.iterrows():
        # Filter to frontal projections only
        if prj_col and pd.notna(row[prj_col]):
            pv = str(row[prj_col]).lower()
            if not any(x in pv for x in ['frontal', 'pa', 'ap']):
                continue

        fname = str(row[fn_col]).strip()
        if not fname or fname == 'nan':
            continue

        uid = str(row[uid_col_proj]).strip()
        text = uid_to_text.get(uid, '')

        # Build label vector from free-text keyword matching
        labels = np.zeros(len(all_cls), dtype=np.float32)
        for kw, cls_name in KEYWORD_MAP.items():
            if kw in text:
                labels[all_cls.index(cls_name)] = 1.0

        # Verify image exists
        img_path = os.path.join(image_dir, fname)
        if not os.path.exists(img_path):
            if not os.path.exists(img_path + '.png'):
                continue

        records.append({'filename': fname, 'labels': labels})

    print(f"  Valid frontal records: {len(records):,}")
    pos_counts = np.array([r['labels'] for r in records]).sum(0)
    print(f"  Label distribution (pos counts per class):")
    for i, cn in enumerate(all_cls):
        pct = pos_counts[i] / max(len(records), 1) * 100
        print(f"    {cn:<22}: {int(pos_counts[i]):>4}  ({pct:.1f}%)")
    return records

    print(f"  Valid frontal records: {len(records):,}")
    pos_counts = np.array([r['labels'] for r in records]).sum(0)
    print(f"  Label distribution (pos counts per class):")
    for i, cn in enumerate(all_cls):
        print(f"    {cn:<22}: {int(pos_counts[i]):>4}")
    return records

print("✓ IndianaChestXrayDataset + build_indiana_records defined")

✓ IndianaChestXrayDataset + build_indiana_records defined


## Cell 6 — DataLoaders
> Same `random_state=42` as baseline — identical train/val split.

In [6]:
# ── Kaggle dataset path or local fallback ─────────────────────────────
IN_KAGGLE = Path('/kaggle').exists()
if IN_KAGGLE:
    DATASET_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university'
    IMAGE_DIR   = os.path.join(DATASET_DIR, 'images', 'images_normalized')
    PROJ_CSV    = os.path.join(DATASET_DIR, 'indiana_projections.csv')
    REPORTS_CSV = os.path.join(DATASET_DIR, 'indiana_reports.csv')
    CHECKPOINT_DIR = '/kaggle/working/checkpoints'
else:
    DATASET_DIR = str(Path.cwd() / 'chest-xrays-indiana-university')
    IMAGE_DIR   = os.path.join(DATASET_DIR, 'images', 'images_normalized')
    PROJ_CSV    = os.path.join(DATASET_DIR, 'indiana_projections.csv')
    REPORTS_CSV = os.path.join(DATASET_DIR, 'indiana_reports.csv')
    CHECKPOINT_DIR = os.path.join(Path.cwd(), 'checkpoints')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Kaggle path check:")
for lbl, p in [('Dataset dir',  DATASET_DIR),
                ('Image dir',    IMAGE_DIR),
                ('Proj CSV',     PROJ_CSV),
                ('Reports CSV',  REPORTS_CSV)]:
    print(f"  {'✓' if os.path.exists(p) else '✗'}  {lbl}: {p}")

# ── Build records from Indiana dataset or synthetic fallback ──────────
print("\nBuilding dataset records...")
if os.path.exists(PROJ_CSV) and os.path.exists(IMAGE_DIR):
    all_records = build_indiana_records(PROJ_CSV, IMAGE_DIR)
    USING_SYNTHETIC_DATA = False
else:
    USING_SYNTHETIC_DATA = True
    print("⚠ Indiana dataset not found — using a tiny synthetic fallback so the notebook can execute locally.")
    rng = np.random.default_rng(42)
    all_records = []
    for i in range(12):
        labels = np.zeros(len(IndianaChestXrayDataset.ALL_CLASSES), dtype=np.float32)
        if i < 6:
            labels[rng.integers(0, len(labels))] = 1.0
        all_records.append({'filename': f'synthetic_{i}.png', 'labels': labels})
    print(f"  Synthetic records: {len(all_records)} rows")

# ── Train / Val split (80/20, stratified by any-positive label) ──────
random.seed(42); np.random.seed(42)
has_finding = [int(r['labels'].sum() > 0) for r in all_records]
train_recs, val_recs = train_test_split(
    all_records, test_size=0.20, random_state=42,
    stratify=has_finding if len(set(has_finding)) > 1 else None
)
print(f"\n  Train split: {len(train_recs):,} images")
print(f"  Val   split: {len(val_recs):,} images")

# ── ImageNet normalisation ────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = IndianaChestXrayDataset(IMAGE_DIR, train_recs, train_transform)
val_dataset   = IndianaChestXrayDataset(IMAGE_DIR, val_recs,   val_transform)

# ── DataLoaders ───────────────────────────────────────────────────────
BATCH_SIZE  = 4 if USING_SYNTHETIC_DATA else 32
ACCUM_STEPS = 1 if USING_SYNTHETIC_DATA else 4
NUM_WORKERS = 0 if USING_SYNTHETIC_DATA else 2   # local fallback should stay simple

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"\n  Train : {len(train_dataset):,} | {len(train_loader):,} batches")
print(f"  Val   : {len(val_dataset):,} | {len(val_loader):,} batches")
print(f"  Physical batch={BATCH_SIZE} | Effective batch={BATCH_SIZE*ACCUM_STEPS}")

imgs, lbls = next(iter(train_loader))
print(f"\nSanity: images={imgs.shape}  labels={lbls.shape}")
print(f"  pixel min={imgs.min():.3f}  max={imgs.max():.3f}")
print(f"  pos labels/sample={lbls.sum(1).mean():.2f}")
print("✓ DataLoaders ready")

Kaggle path check:
  ✓  Dataset dir: /kaggle/input/datasets/raddar/chest-xrays-indiana-university
  ✓  Image dir: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized
  ✓  Proj CSV: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv
  ✓  Reports CSV: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv

Building dataset records...
  Projections CSV: 7,466 rows, columns: ['uid', 'filename', 'projection']
  Reports CSV    : 3,851 rows, columns: ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']
  Proj  → uid='uid'  filename='filename'  projection='projection'
  Rep   → uid='uid'  findings='findings'  impression='impression'
  uid→text entries: 3,851
  Valid frontal records: 3,818
  Label distribution (pos counts per class):
    Atelectasis           :  364  (9.5%)
    Consolidation         : 1212  (31.7%)
    Infiltration          :  412  (10.8%)
    Pneu

## Cell 7 — Load Model (SpatialMamba Stage 4, Head → 14 Classes)

In [7]:
print("Loading MambaVision-T with SpatialMamba Layer 4...")

# FIX (Bug 5): use pretrained=True to load ImageNet weights for stages 1-3.
# Training from random init on a small dataset caused flat loss/accuracy.
model = mamba_vision_T(pretrained=True, num_classes=1000, depths=[1, 3, 8, 3])
print("\u2713 Model loaded WITH pretrained ImageNet weights (stages 1\u20133 initialised)")

assert type(model.levels[3]).__name__ == 'SpatialMambaLayer', \
    f"Got {type(model.levels[3]).__name__} \u2014 expected SpatialMambaLayer"
print(f"\u2713 Stage 4 = {type(model.levels[3]).__name__}  \u2190 REPLACED")

# Replace classification head for 14-class multi-label task
in_features = model.head.in_features
model.head  = nn.Linear(in_features, 14, bias=True)
nn.init.trunc_normal_(model.head.weight, std=0.02)
nn.init.zeros_(model.head.bias)

# All parameters trainable — differential LR (Cell 9) handles backbone vs head
for p in model.parameters(): p.requires_grad = True
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
stage4_params    = sum(p.numel() for p in model.levels[3].parameters())

EXP_NAME = f'spatial_mamba_indiana_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print(f"Stage 1 : {type(model.levels[0]).__name__}")
print(f"Stage 2 : {type(model.levels[1]).__name__}")
print(f"Stage 3 : {type(model.levels[2]).__name__}")
print(f"Stage 4 : {type(model.levels[3]).__name__}  \u2190 REPLACED")
print(f"\nTotal params     : {total_params:,}")
print(f"Stage 4 params   : {stage4_params:,}  (SpatialMambaLayer only)")
print(f"Model size FP32  : {total_params*4/1024**2:.2f} MB")
print(f"\n\u2713 Model on {device}  |  Experiment: {EXP_NAME}")
print(f"  Mamba backend  : {'mamba-ssm CUDA' if USE_MAMBA_SSM_CUDA else 'pure-PyTorch GRU (P100-safe)'}")


Loading MambaVision-T with SpatialMamba Layer 4...


100%|██████████| 364M/364M [00:00<00:00, 417MB/s] 


The model and loaded state dict do not match exactly

size mismatch for levels.3.blocks.0.norm1.weight: copying a param with shape torch.Size([640]) from checkpoint, the shape in current model is torch.Size([560]).
size mismatch for levels.3.blocks.0.norm1.bias: copying a param with shape torch.Size([640]) from checkpoint, the shape in current model is torch.Size([560]).
size mismatch for levels.3.blocks.0.norm2.weight: copying a param with shape torch.Size([640]) from checkpoint, the shape in current model is torch.Size([560]).
size mismatch for levels.3.blocks.0.norm2.bias: copying a param with shape torch.Size([640]) from checkpoint, the shape in current model is torch.Size([560]).
size mismatch for levels.3.blocks.1.norm1.weight: copying a param with shape torch.Size([640]) from checkpoint, the shape in current model is torch.Size([560]).
size mismatch for levels.3.blocks.1.norm1.bias: copying a param with shape torch.Size([640]) from checkpoint, the shape in current model is torch

## Cell 8 — FLOPs & Parameter Count
> Restored wider 7/8 stage width so params sit near the 25M target.

In [8]:
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
GFLOPS = 0.0

if FLOPS_AVAILABLE:
    try:
        with FlopCounterMode(model, display=False) as fc:
            with torch.no_grad():
                model(dummy_input)
        flops  = fc.get_total_flops()
        GFLOPS = flops / 1e9
        flops_method = "fvcore (exact)"
    except Exception as e:
        print(f"  fvcore measurement failed: {e}")
        FLOPS_AVAILABLE = False

if not FLOPS_AVAILABLE and THOP_AVAILABLE:
    try:
        flops, _ = thop_profile(model, inputs=(dummy_input,), verbose=False)
        GFLOPS   = flops / 1e9
        flops_method = "thop (approx)"
    except Exception as e:
        print(f"  thop measurement failed: {e}")

if GFLOPS == 0.0:
    GFLOPS = 4.5
    flops_method = "paper estimate"

print(f"FLOPs method     : {flops_method}")
print(f"GFLOPs / image   : {GFLOPS:.4f}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Frozen params    : {frozen_params:,}")
print(f"Model size FP32  : {total_params*4/1024**2:.2f} MB")

FLOPs method     : thop (approx)
GFLOPs / image   : 3.8323
Total params     : 20,052,302
Trainable params : 20,052,302
Frozen params    : 0
Model size FP32  : 76.49 MB


## Cell 9 — Loss / Optimiser / Scheduler
> **Identical to baseline** — dynamic pos_weight from same training split.

In [9]:
NUM_EPOCHS    = 30
LR_BACKBONE   = 1e-4
LR_HEAD       = 5e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
WARMUP_EPOCHS = 5
USE_AMP       = True and torch.cuda.is_available()   # AMP only on CUDA
PATIENCE      = 7
MIN_DELTA     = 1e-4

if USING_SYNTHETIC_DATA:
    NUM_EPOCHS    = 1
    WARMUP_EPOCHS = 0
    PATIENCE      = 1
    print("⚠ Synthetic fallback detected — using a single quick epoch for local validation")

# pos_weight = N_neg/N_pos per class
# Recompute from actual Indiana dataset label distribution
all_labels = np.array([r['labels'] for r in train_recs])
n_pos = all_labels.sum(0).clip(min=1)
n_neg = len(all_labels) - n_pos
pw    = (n_neg / n_pos).clip(max=300).astype(np.float32)
pos_weight = torch.tensor(pw, device=device)
print(f"pos_weight range: [{pw.min():.1f}, {pw.max():.1f}]")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

param_groups = [
    {'params': [p for n,p in model.named_parameters() if 'head' not in n],
     'lr': LR_BACKBONE, 'weight_decay': WEIGHT_DECAY},
    {'params': list(model.head.parameters()),
     'lr': LR_HEAD, 'weight_decay': 0.0},
]
optimizer = optim.AdamW(param_groups, betas=(0.9,0.999), eps=1e-8)

if WARMUP_EPOCHS > 0:
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS-WARMUP_EPOCHS, eta_min=1e-7)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
else:
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda _: 1.0)

scaler = GradScaler('cuda', enabled=USE_AMP) if torch.cuda.is_available() else GradScaler(enabled=False)

CLASS_NAMES = [
    'Atelectasis','Consolidation','Infiltration','Pneumothorax',
    'Edema','Emphysema','Fibrosis','Effusion','Pneumonia',
    'Pleural_Thickening','Cardiomegaly','Nodule','Mass','Hernia'
]
NUM_CLASSES = len(CLASS_NAMES)

SAVE_DIR = os.path.join(CHECKPOINT_DIR, EXP_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\nExperiment      : {EXP_NAME}")
print(f"Checkpoint dir  : {SAVE_DIR}")
print(f"Backbone LR     : {LR_BACKBONE:.2e}")
print(f"Head LR         : {LR_HEAD:.2e}")
print(f"Warmup          : {WARMUP_EPOCHS} ep → Cosine to ep {NUM_EPOCHS}")
print(f"Mixed precision : {USE_AMP}")
print("✓ Loss / Optimiser / Scheduler ready")

pos_weight range: [0.4, 108.1]

Experiment      : spatial_mamba_indiana_20260503_213433
Checkpoint dir  : /kaggle/working/checkpoints/spatial_mamba_indiana_20260503_213433
Backbone LR     : 1.00e-04
Head LR         : 5.00e-04
Warmup          : 5 ep → Cosine to ep 30
Mixed precision : True
✓ Loss / Optimiser / Scheduler ready


## Cell 10 — Training & Validation Functions
> **Identical to baseline** — same 13 metrics.

In [10]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val=self.avg=self.sum=self.count=0
    def update(self,val,n=1):
        self.val=val; self.sum+=val*n; self.count+=n; self.avg=self.sum/self.count



def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch, accum=4):
    model.train()
    meter = AverageMeter()
    optimizer.zero_grad()
    for i, (imgs, tgts) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP) if torch.cuda.is_available() else nullcontext():
            out  = model(imgs)
            loss = criterion(out, tgts) / accum
        scaler.scale(loss).backward()
        if (i+1) % accum == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        meter.update(loss.item()*accum, imgs.size(0))
        if (i+1) % max(1, len(loader)//3) == 0 or (i+1) == len(loader):
            print(f"  Ep[{epoch}][{i+1}/{len(loader)}] loss={meter.avg:.4f}"
                  f"  lr={optimizer.param_groups[0]['lr']:.2e}", flush=True)
    return meter.avg



def validate(model, loader, criterion, device):
    model.eval()
    meter = AverageMeter()
    probs_all, tgts_all = [], []
    with torch.no_grad():
        for imgs, tgts in loader:
            imgs = imgs.to(device, non_blocking=True)
            tgts = tgts.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP) if torch.cuda.is_available() else nullcontext():
                out  = model(imgs)
                loss = criterion(out, tgts)
            meter.update(loss.item(), imgs.size(0))
            probs_all.append(torch.sigmoid(out).cpu().numpy())
            tgts_all.append(tgts.cpu().numpy())

    probs = np.concatenate(probs_all)
    tgts  = np.concatenate(tgts_all)
    preds = (probs >= 0.5).astype(int)

    acc     = (preds == tgts).mean()
    emr     = (preds == tgts).all(axis=1).mean()
    hl      = hamming_loss(tgts, preds)
    prec_mi = precision_score(tgts, preds, average='micro',    zero_division=0)
    rec_mi  = recall_score(   tgts, preds, average='micro',    zero_division=0)
    f1_mi   = f1_score(       tgts, preds, average='micro',    zero_division=0)
    f1_ma   = f1_score(       tgts, preds, average='macro',    zero_division=0)
    f1_wt   = f1_score(       tgts, preds, average='weighted', zero_division=0)
    f1_pc   = f1_score(       tgts, preds, average=None,       zero_division=0)
    prec_pc = precision_score(tgts, preds, average=None,       zero_division=0)
    rec_pc  = recall_score(   tgts, preds, average=None,       zero_division=0)

    spec_pc = np.zeros(tgts.shape[1])
    for ci in range(tgts.shape[1]):
        tn = ((preds[:,ci]==0)&(tgts[:,ci]==0)).sum()
        fp = ((preds[:,ci]==1)&(tgts[:,ci]==0)).sum()
        spec_pc[ci] = tn/(tn+fp) if (tn+fp)>0 else 0.0

    mcc_pc   = np.array([matthews_corrcoef(tgts[:,ci], preds[:,ci])
                          for ci in range(tgts.shape[1])])
    mcc_mean = float(np.mean(mcc_pc))

    try:
        auc_ma = roc_auc_score(tgts, probs, average='macro')
        auc_mi = roc_auc_score(tgts, probs, average='micro')
        auc_pc = roc_auc_score(tgts, probs, average=None)
    except Exception:
        auc_ma = auc_mi = 0.0; auc_pc = np.zeros(NUM_CLASSES)

    try:
        map_score = average_precision_score(tgts, probs, average='macro')
        ap_pc     = average_precision_score(tgts, probs, average=None)
    except Exception:
        map_score = 0.0; ap_pc = np.zeros(NUM_CLASSES)

    return {
        'loss': meter.avg, 'accuracy': float(acc),
        'emr': float(emr),  'hamming': float(hl),
        'precision': float(prec_mi), 'recall': float(rec_mi),
        'f1_micro': float(f1_mi),    'f1_macro': float(f1_ma),
        'f1_weighted': float(f1_wt), 'auc_macro': float(auc_ma),
        'auc_micro': float(auc_mi),  'map': float(map_score),
        'mcc': float(mcc_mean),
        'f1_per': f1_pc,   'prec_per': prec_pc, 'rec_per':  rec_pc,
        'spec_per': spec_pc,'auc_per': auc_pc,   'ap_per':   ap_pc,
        'mcc_per': mcc_pc,  'probs': probs,       'targets':  tgts,
    }

print("✓ train_epoch + validate defined")

✓ train_epoch + validate defined


## Cell 11 — Training Loop
> GPU peak training memory tracked every epoch.

In [11]:
from IPython.display import clear_output

history = {k: [] for k in [
    'epoch','train_loss','val_loss',
    'accuracy','emr','hamming',
    'precision','recall',
    'f1_micro','f1_macro','f1_weighted',
    'auc_macro','auc_micro','map','mcc',
    'lr_backbone','lr_head'
]}

best_f1 = 0.0; patience_cnt = 0; PEAK_TRAIN_MB = 0.0

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

LOG_FILE = os.path.join(SAVE_DIR, 'training_log.txt')
log_f = open(LOG_FILE, 'w')
print(f"Training [BASELINE] — {NUM_EPOCHS} epochs")
log_f.write(f"Training [BASELINE] — {NUM_EPOCHS} epochs\n")

for epoch in range(1, NUM_EPOCHS+1):
    t_loss = train_epoch(model, train_loader, criterion, optimizer,
                         scaler, device, epoch, accum=ACCUM_STEPS)
    if torch.cuda.is_available():
        peak_now = torch.cuda.max_memory_allocated()/1024**2
        if peak_now > PEAK_TRAIN_MB: PEAK_TRAIN_MB = peak_now
    v = validate(model, val_loader, criterion, device)
    scheduler.step()
    lr_bb = optimizer.param_groups[0]['lr']
    lr_hd = optimizer.param_groups[1]['lr']

    clear_output(wait=True)
    print(f"Training [BASELINE] — {NUM_EPOCHS} epochs")
    print(f"{'='*70}")
    print(f"  Epoch {epoch}/{NUM_EPOCHS}  lr_bb={lr_bb:.2e}  patience={patience_cnt}/{PATIENCE}")
    print(f"{'='*70}")
    print(f"  Train loss   : {t_loss:.4f}")
    print(f"  Val   loss   : {v['loss']:.4f}")
    print(f"  Accuracy     : {v['accuracy']:.4f}")
    print(f"  EMR          : {v['emr']:.4f}")
    print(f"  Hamming      : {v['hamming']:.4f}  ↓")
    print(f"  Precision    : {v['precision']:.4f}")
    print(f"  Recall       : {v['recall']:.4f}")
    print(f"  F1 micro     : {v['f1_micro']:.4f}")
    print(f"  F1 macro     : {v['f1_macro']:.4f}")
    print(f"  F1 weighted  : {v['f1_weighted']:.4f}")
    print(f"  AUC-ROC mac  : {v['auc_macro']:.4f}")
    print(f"  AUC-ROC mic  : {v['auc_micro']:.4f}")
    print(f"  mAP (AUC-PR) : {v['map']:.4f}")
    print(f"  MCC          : {v['mcc']:.4f}")
    if torch.cuda.is_available():
        print(f"  Peak VRAM    : {PEAK_TRAIN_MB:.1f} MB")

    # Running epoch table
    print(f"\n{'─'*75}")
    print(f"  {'Ep':>3} {'TrLoss':>7} {'VaLoss':>7} {'Acc':>6} {'F1mic':>6} {'F1mac':>6} {'AUC':>6} {'mAP':>6} {'MCC':>6}")
    print(f"{'─'*75}")
    for ep_i in range(len(history['epoch'])):
        print(f"  {history['epoch'][ep_i]:>3} {history['train_loss'][ep_i]:>7.4f} "
              f"{history['val_loss'][ep_i]:>7.4f} {history['accuracy'][ep_i]:>6.4f} "
              f"{history['f1_micro'][ep_i]:>6.4f} {history['f1_macro'][ep_i]:>6.4f} "
              f"{history['auc_macro'][ep_i]:>6.4f} {history['map'][ep_i]:>6.4f} "
              f"{history['mcc'][ep_i]:>6.4f}")
    # Print current epoch last
    print(f"  {epoch:>3} {t_loss:>7.4f} {v['loss']:>7.4f} {v['accuracy']:>6.4f} "
          f"{v['f1_micro']:>6.4f} {v['f1_macro']:>6.4f} "
          f"{v['auc_macro']:>6.4f} {v['map']:>6.4f} {v['mcc']:>6.4f}  ← current")

    for k,val_ in [
        ('epoch',epoch),('train_loss',t_loss),('val_loss',v['loss']),
        ('accuracy',v['accuracy']),('emr',v['emr']),('hamming',v['hamming']),
        ('precision',v['precision']),('recall',v['recall']),
        ('f1_micro',v['f1_micro']),('f1_macro',v['f1_macro']),
        ('f1_weighted',v['f1_weighted']),('auc_macro',v['auc_macro']),
        ('auc_micro',v['auc_micro']),('map',v['map']),('mcc',v['mcc']),
        ('lr_backbone',lr_bb),('lr_head',lr_hd),
    ]:
        history[k].append(val_)

    log_line = (f"Ep{epoch:03d} | loss={t_loss:.4f} | val={v['loss']:.4f} | "
                f"acc={v['accuracy']:.4f} | f1={v['f1_micro']:.4f} | "
                f"auc={v['auc_macro']:.4f} | map={v['map']:.4f} | mcc={v['mcc']:.4f}")
    log_f.write(log_line + "\n"); log_f.flush()

    if v['f1_micro'] > best_f1 + MIN_DELTA:
        best_f1 = v['f1_micro']; patience_cnt = 0
        best_path = os.path.join(SAVE_DIR, 'best_model.pth')
        torch.save({
            'epoch': epoch, 'best_f1': best_f1,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_metrics': {k: v[k] for k in [
                'loss','accuracy','emr','hamming','precision','recall',
                'f1_micro','f1_macro','f1_weighted',
                'auc_macro','auc_micro','map','mcc']},
            'history': history,
        }, best_path)
        print(f"  ★ Best model saved  F1={best_f1:.4f}")
    else:
        patience_cnt += 1

    if epoch % 5 == 0:
        ckpt = os.path.join(SAVE_DIR, f'checkpoint_ep{epoch:03d}.pth')
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                     'history':history}, ckpt)

    if patience_cnt >= PATIENCE:
        print(f"  Early stopping at epoch {epoch}")
        break

log_f.close()
print(f"\nTraining done | Best F1 (micro): {best_f1:.4f}")
print(f"Peak training VRAM: {PEAK_TRAIN_MB:.2f} MB")

Training [BASELINE] — 30 epochs
  Epoch 30/30  lr_bb=1.00e-07  patience=4/7
  Train loss   : 0.7179
  Val   loss   : 0.9374
  Accuracy     : 0.7020
  EMR          : 0.0380
  Hamming      : 0.2980  ↓
  Precision    : 0.2994
  Recall       : 0.6029
  F1 micro     : 0.4002
  F1 macro     : 0.2872
  F1 weighted  : 0.5300
  AUC-ROC mac  : 0.7016
  AUC-ROC mic  : 0.7332
  mAP (AUC-PR) : 0.3261
  MCC          : 0.1673
  Peak VRAM    : 1391.8 MB

───────────────────────────────────────────────────────────────────────────
   Ep  TrLoss  VaLoss    Acc  F1mic  F1mac    AUC    mAP    MCC
───────────────────────────────────────────────────────────────────────────
    1  1.1578  1.1020 0.4885 0.2360 0.1767 0.5074 0.1714 0.0067
    2  1.1489  1.0797 0.5896 0.2855 0.2084 0.5955 0.2145 0.0575
    3  1.1205  1.0405 0.5476 0.2452 0.2041 0.6431 0.2382 0.0820
    4  1.0799  0.9913 0.5303 0.2477 0.2060 0.6813 0.2616 0.0983
    5  1.0458  0.9566 0.6354 0.3038 0.2343 0.6863 0.2649 0.1284
    6  1.0214  0.9790

## Cell 12 — Load Best Model

In [12]:
best_path = os.path.join(SAVE_DIR, 'best_model.pth')
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✓ Best model loaded — Epoch {ckpt['epoch']}  F1={ckpt['best_f1']:.4f}")
else:
    print("⚠ No best_model.pth — using current weights")
model.eval()
print("  eval() mode set")

✓ Best model loaded — Epoch 25  F1=0.4064
  eval() mode set


## Cell 13 — Inference Time
> After training, warm GPU, 100 benchmark runs.

In [13]:
WARMUP_RUNS = 10; BENCH_RUNS = 100
model.eval()
single_inp = torch.randn(1, 3, 224, 224).to(device)
batch_inp  = torch.randn(BATCH_SIZE, 3, 224, 224).to(device)

with torch.no_grad():
    for _ in range(WARMUP_RUNS): _ = model(single_inp)
if torch.cuda.is_available(): torch.cuda.synchronize()

times_single = []
with torch.no_grad():
    for _ in range(BENCH_RUNS):
        t0 = time.perf_counter()
        _ = model(single_inp)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times_single.append((time.perf_counter()-t0)*1000)

T_MEAN = float(np.mean(times_single)); T_STD  = float(np.std(times_single))
T_MIN  = float(np.min(times_single));  T_MAX  = float(np.max(times_single))
T_P95  = float(np.percentile(times_single, 95))
FPS_SINGLE = 1000.0 / T_MEAN

print(f"Inference — single image ({BENCH_RUNS} runs, warm GPU):")
print(f"  Mean : {T_MEAN:.3f} ms  |  Std : {T_STD:.3f} ms")
print(f"  Min  : {T_MIN:.3f} ms  |  P95 : {T_P95:.3f} ms")
print(f"  FPS  : {FPS_SINGLE:.1f} imgs/sec")

with torch.no_grad():
    for _ in range(WARMUP_RUNS): _ = model(batch_inp)
if torch.cuda.is_available(): torch.cuda.synchronize()

times_batch = []
with torch.no_grad():
    for _ in range(BENCH_RUNS):
        t0 = time.perf_counter()
        _ = model(batch_inp)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times_batch.append((time.perf_counter()-t0)*1000)

BT_MEAN    = float(np.mean(times_batch))
BT_PER_IMG = BT_MEAN / BATCH_SIZE
FPS_BATCH  = BATCH_SIZE * 1000.0 / BT_MEAN
print(f"\nInference — batch={BATCH_SIZE}: {BT_MEAN:.3f} ms total  |  {BT_PER_IMG:.3f} ms/img  |  {FPS_BATCH:.1f} FPS")
print("✓ Inference benchmark complete")

Inference — single image (100 runs, warm GPU):
  Mean : 10.647 ms  |  Std : 1.366 ms
  Min  : 9.949 ms  |  P95 : 13.964 ms
  FPS  : 93.9 imgs/sec

Inference — batch=32: 119.225 ms total  |  3.726 ms/img  |  268.4 FPS
✓ Inference benchmark complete


## Cell 14 — GPU Memory
> Inference peak (after training) + training peak (Cell 11).

In [16]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    model.eval()
    with torch.no_grad():
        _ = model(torch.randn(1,3,224,224).to(device))
    torch.cuda.synchronize()
    INFER_ALLOC_MB = torch.cuda.memory_allocated()   / 1024**2
    INFER_CACHE_MB = torch.cuda.memory_reserved()    / 1024**2
    INFER_PEAK_MB  = torch.cuda.max_memory_allocated()/ 1024**2
    print(f"GPU Memory — INFERENCE (1 image):")
    print(f"  Allocated : {INFER_ALLOC_MB:.2f} MB  |  Cached : {INFER_CACHE_MB:.2f} MB")
    print(f"  Peak      : {INFER_PEAK_MB:.2f} MB   |  Util   : {INFER_PEAK_MB/TOTAL_VRAM_MB*100:.1f}%")
    print(f"\nGPU Memory — TRAINING PEAK:")
    print(f"  Peak      : {PEAK_TRAIN_MB:.2f} MB   |  Util   : {PEAK_TRAIN_MB/TOTAL_VRAM_MB*100:.1f}%")
    print(f"  Total VRAM: {TOTAL_VRAM_MB:.2f} MB")
else:
    INFER_ALLOC_MB = INFER_CACHE_MB = INFER_PEAK_MB = 0.0
    print("⚠ CUDA not available")
print("✓ GPU memory profiled")

GPU Memory — INFERENCE (1 image):
  Allocated : 594.79 MB  |  Cached : 788.00 MB
  Peak      : 599.72 MB   |  Util   : 4.0%

GPU Memory — TRAINING PEAK:
  Peak      : 1391.82 MB   |  Util   : 9.3%
  Total VRAM: 14912.69 MB
✓ GPU memory profiled


## Cell 15 — Final Evaluation — All 13 Metrics

In [17]:
print("Running final evaluation...")
final = validate(model, val_loader, criterion, device)

print(f"\n{'─'*62}")
print(f"  {'METRIC':<30} {'VALUE':>10}")
print(f"  {'─'*42}")
for lbl, key in [
    ('Loss',                    'loss'),
    ('Accuracy (element-wise)', 'accuracy'),
    ('Exact Match Ratio (EMR)', 'emr'),
    ('Hamming Loss ↓',          'hamming'),
    ('Precision (micro)',       'precision'),
    ('Recall (micro)',          'recall'),
    ('F1 (micro)',              'f1_micro'),
    ('F1 (macro)',              'f1_macro'),
    ('F1 (weighted)',           'f1_weighted'),
    ('AUC-ROC (macro)',         'auc_macro'),
    ('AUC-ROC (micro)',         'auc_micro'),
    ('mAP / AUC-PR (macro)',    'map'),
    ('MCC (mean)',              'mcc'),
]:
    print(f"  {lbl:<30} {final[key]:>10.4f}")
print(f"  {'─'*42}")

print(f"\nPer-class (F1 / AUC / AP / Prec / Rec / Spec / MCC):")
print(f"  {'Class':<22} {'F1':>6} {'AUC':>6} {'AP':>6} {'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6}")
print(f"  {'─'*64}")
for i, cn in enumerate(CLASS_NAMES):
    print(f"  {cn:<22} {final['f1_per'][i]:>6.3f} {final['auc_per'][i]:>6.3f}"
          f" {final['ap_per'][i]:>6.3f} {final['prec_per'][i]:>6.3f}"
          f" {final['rec_per'][i]:>6.3f} {final['spec_per'][i]:>6.3f}"
          f" {final['mcc_per'][i]:>6.3f}")

Running final evaluation...

──────────────────────────────────────────────────────────────
  METRIC                              VALUE
  ──────────────────────────────────────────
  Loss                               0.9328
  Accuracy (element-wise)            0.7127
  Exact Match Ratio (EMR)            0.0419
  Hamming Loss ↓                     0.2873
  Precision (micro)                  0.3081
  Recall (micro)                     0.5967
  F1 (micro)                         0.4064
  F1 (macro)                         0.2860
  F1 (weighted)                      0.5315
  AUC-ROC (macro)                    0.7037
  AUC-ROC (micro)                    0.7392
  mAP / AUC-PR (macro)               0.3217
  MCC (mean)                         0.1670
  ──────────────────────────────────────────

Per-class (F1 / AUC / AP / Prec / Rec / Spec / MCC):
  Class                      F1    AUC     AP   Prec    Rec   Spec    MCC
  ────────────────────────────────────────────────────────────────
  Atele

## Cell 16 — Training Curves (3×3)

In [18]:
ep = history['epoch']
fig, axes = plt.subplots(3, 3, figsize=(20, 15))

def plot(ax, keys, title, logy=False):
    styles=[('o-','steelblue'),('s-','coral'),('^-','green'),('d-','purple')]
    for (key,lbl),(mk,col) in zip(keys,styles):
        ax.plot(ep, history[key], mk, ms=3, color=col, label=lbl)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    if logy: ax.set_yscale('log')

plot(axes[0,0],[('train_loss','Train'),('val_loss','Val')],'Loss')
plot(axes[0,1],[('f1_micro','micro'),('f1_macro','macro'),('f1_weighted','weighted')],'F1 Score')
plot(axes[0,2],[('auc_macro','AUC-ROC mac'),('auc_micro','AUC-ROC mic'),('map','mAP')],'AUC & mAP')
plot(axes[1,0],[('precision','Precision'),('recall','Recall')],'Precision & Recall')
plot(axes[1,1],[('accuracy','Accuracy'),('emr','Exact Match')],'Accuracy & EMR')
plot(axes[1,2],[('mcc','MCC'),('hamming','Hamming↓')],'MCC & Hamming Loss')
plot(axes[2,0],[('lr_backbone','Backbone'),('lr_head','Head')],'Learning Rate',logy=True)

axes[2,1].scatter(history['recall'],history['precision'],
                  c=history['epoch'],cmap='viridis',s=30,zorder=3)
axes[2,1].set_xlabel('Recall'); axes[2,1].set_ylabel('Precision')
axes[2,1].set_title('Precision vs Recall (epochs)',fontweight='bold'); axes[2,1].grid(alpha=.3)

axes[2,2].scatter(history['f1_micro'],history['auc_macro'],
                  c=history['epoch'],cmap='plasma',s=30,zorder=3)
axes[2,2].set_xlabel('F1 micro'); axes[2,2].set_ylabel('AUC-ROC')
axes[2,2].set_title('F1 vs AUC-ROC (epochs)',fontweight='bold'); axes[2,2].grid(alpha=.3)

plt.suptitle('MambaVision-T SpatialMamba Stage 4 | Indiana Chest X-ray (Kaggle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show(); print("✓ Training curves saved")

✓ Training curves saved


## Cell 17 — Per-Class Visualisation

In [19]:
fig, axes = plt.subplots(3, 3, figsize=(24, 20))

def hbar(ax, vals, title, color):
    idx = np.argsort(vals)
    vs  = vals[idx]; ns = [CLASS_NAMES[i] for i in idx]
    bars = ax.barh(ns, vs, color=color, edgecolor='white', alpha=0.85)
    ax.set_xlim([0,1]); ax.set_title(title, fontweight='bold', fontsize=11)
    mv = np.mean(vals)
    ax.axvline(mv, color='black', ls='--', lw=1.2, label=f'Mean={mv:.3f}')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, vs):
        ax.text(min(v+.01, 0.93), bar.get_y()+bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)
    ax.grid(axis='x', alpha=.3)

hbar(axes[0,0],final['f1_per'],  'Per-Class F1',         'steelblue')
hbar(axes[0,1],final['auc_per'], 'Per-Class AUC-ROC',    'coral')
hbar(axes[0,2],final['ap_per'],  'Per-Class AP (AUC-PR)','darkorange')
hbar(axes[1,0],final['prec_per'],'Per-Class Precision',  'mediumseagreen')
hbar(axes[1,1],final['rec_per'], 'Per-Class Recall',     'mediumpurple')
hbar(axes[1,2],final['spec_per'],'Per-Class Specificity','teal')
hbar(axes[2,0],final['mcc_per'], 'Per-Class MCC',        'saddlebrown')

metric_names = ['F1','AUC-ROC','AP','Precision','Recall','Specificity','MCC']
metric_vals  = [np.mean(final['f1_per']),np.mean(final['auc_per']),
                np.mean(final['ap_per']),np.mean(final['prec_per']),
                np.mean(final['rec_per']),np.mean(final['spec_per']),
                np.clip(np.mean(final['mcc_per']),0,1)]
angles = np.linspace(0,2*np.pi,len(metric_names),endpoint=False).tolist()
vr = metric_vals+[metric_vals[0]]; ar = angles+[angles[0]]
axes[2,1].remove()
ax_r = fig.add_subplot(3,3,8,polar=True)
ax_r.plot(ar,vr,'o-',lw=2,color='steelblue'); ax_r.fill(ar,vr,alpha=0.25,color='steelblue')
ax_r.set_xticks(angles); ax_r.set_xticklabels(metric_names,fontsize=9)
ax_r.set_ylim(0,1); ax_r.set_title('Mean Metrics Radar',fontweight='bold',pad=15)

hdata = np.array([final['f1_per'],final['auc_per'],final['ap_per'],
                  final['prec_per'],final['rec_per'],final['spec_per'],
                  np.clip(final['mcc_per'],0,1)])
ax_h = axes[2,2]
im = ax_h.imshow(hdata, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax_h.set_xticks(range(len(CLASS_NAMES)))
ax_h.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
ax_h.set_yticks(range(7)); ax_h.set_yticklabels(metric_names, fontsize=9)
ax_h.set_title('Per-Class Heatmap', fontweight='bold')
plt.colorbar(im, ax=ax_h, fraction=0.03)
for r in range(7):
    for col in range(len(CLASS_NAMES)):
        ax_h.text(col,r,f'{hdata[r,col]:.2f}',ha='center',va='center',fontsize=6)

plt.suptitle('MambaVision-T SpatialMamba Stage 4 | Indiana Chest X-ray (Kaggle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'per_class_metrics.png'),dpi=150,bbox_inches='tight')
plt.show(); print("✓ Per-class visualisation saved")

✓ Per-class visualisation saved


## Cell 18 — Full Benchmark Summary Table

In [20]:
SEP = "─"*65
print("\n"+"="*65)
print(f"  MAMBAVISION-T  |  Indiana Chest X-ray  |  EXPERIMENT")
print(f"  Dataset  : Indiana University Chest X-ray (Kaggle)")
print(f"  Stage 4  : SpatialMambaLayer (4-directional SSM — REPLACED)")
print("="*65)
print(f"\n{SEP}")
print("  MODEL COMPLEXITY")
print(SEP)
print(f"  Total parameters         : {total_params:>14,}")
print(f"  Trainable parameters     : {trainable_params:>14,}")
print(f"  Frozen parameters        : {frozen_params:>14,}")
print(f"  Model size  (FP32)       : {total_params*4/1024**2:>10.2f} MB")
print(f"  GFLOPs per image         : {GFLOPS:>10.4f}")
print(f"  FLOPs method             : {flops_method}")
print(f"\n{SEP}")
print("  INFERENCE TIME  (warm GPU, 100 runs)")
print(SEP)
print(f"  Single image — mean      : {T_MEAN:>8.3f} ms")
print(f"  Single image — std       : {T_STD:>8.3f} ms")
print(f"  Single image — P95       : {T_P95:>8.3f} ms")
print(f"  Single image — FPS       : {FPS_SINGLE:>8.1f} imgs/sec")
print(f"  Batch (bs={BATCH_SIZE}) per img    : {BT_PER_IMG:>8.3f} ms")
print(f"  Batch (bs={BATCH_SIZE}) FPS        : {FPS_BATCH:>8.1f} imgs/sec")
if torch.cuda.is_available():
    print(f"\n{SEP}")
    print("  GPU MEMORY")
    print(SEP)
    print(f"  Training peak VRAM       : {PEAK_TRAIN_MB:>8.2f} MB  (fwd+bwd+optim)")
    print(f"  Inference peak VRAM      : {INFER_PEAK_MB:>8.2f} MB  (fwd only)")
    print(f"  Training utilisation     : {PEAK_TRAIN_MB/TOTAL_VRAM_MB*100:>7.1f}%")
    print(f"  Inference utilisation    : {INFER_PEAK_MB/TOTAL_VRAM_MB*100:>7.1f}%")
    print(f"  Total VRAM               : {TOTAL_VRAM_MB:>8.2f} MB")
print(f"\n{SEP}")
print("  CLASSIFICATION METRICS  (val set, best model)")
print(SEP)
for lbl, key in [
    ('Accuracy (element-wise)', 'accuracy'),
    ('Exact Match Ratio',       'emr'),
    ('Hamming Loss ↓',          'hamming'),
    ('Precision  (micro)',      'precision'),
    ('Recall     (micro)',      'recall'),
    ('F1         (micro)',      'f1_micro'),
    ('F1         (macro)',      'f1_macro'),
    ('F1         (weighted)',   'f1_weighted'),
    ('AUC-ROC    (macro)',      'auc_macro'),
    ('AUC-ROC    (micro)',      'auc_micro'),
    ('mAP / AUC-PR (macro)',    'map'),
    ('MCC        (mean)',       'mcc'),
]:
    print(f"  {lbl:<30} : {final[key]:>8.4f}")
print(f"\n{SEP}")
print("  PER-CLASS METRICS")
print(SEP)
print(f"  {'Class':<22} {'F1':>6} {'AUC':>6} {'AP':>6} {'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6}")
print(f"  {'─'*64}")
for i, cn in enumerate(CLASS_NAMES):
    print(f"  {cn:<22} {final['f1_per'][i]:>6.3f} {final['auc_per'][i]:>6.3f}"
          f" {final['ap_per'][i]:>6.3f} {final['prec_per'][i]:>6.3f}"
          f" {final['rec_per'][i]:>6.3f} {final['spec_per'][i]:>6.3f}"
          f" {final['mcc_per'][i]:>6.3f}")
print("="*65)


  MAMBAVISION-T  |  Indiana Chest X-ray  |  EXPERIMENT
  Dataset  : Indiana University Chest X-ray (Kaggle)
  Stage 4  : SpatialMambaLayer (4-directional SSM — REPLACED)

─────────────────────────────────────────────────────────────────
  MODEL COMPLEXITY
─────────────────────────────────────────────────────────────────
  Total parameters         :     20,052,302
  Trainable parameters     :     20,052,302
  Frozen parameters        :              0
  Model size  (FP32)       :      76.49 MB
  GFLOPs per image         :     3.8323
  FLOPs method             : thop (approx)

─────────────────────────────────────────────────────────────────
  INFERENCE TIME  (warm GPU, 100 runs)
─────────────────────────────────────────────────────────────────
  Single image — mean      :   10.647 ms
  Single image — std       :    1.366 ms
  Single image — P95       :   13.964 ms
  Single image — FPS       :     93.9 imgs/sec
  Batch (bs=32) per img    :    3.726 ms
  Batch (bs=32) FPS        :    268.

## Cell 19 — Save Results (JSON + TXT → /kaggle/working/)

In [21]:
results = {
    'experiment': EXP_NAME, 'type': 'Experiment — SpatialMamba Stage 4',
    'stage4': 'SpatialMambaLayer (4-directional SSM)', 'dataset': 'Indiana University Chest X-ray (Kaggle)',
    'generated': datetime.now().isoformat(),
    'model_complexity': {
        'total_params': int(total_params), 'trainable_params': int(trainable_params),
        'frozen_params': int(frozen_params),
        'model_size_fp32_mb': round(total_params*4/1024**2,4),
        'gflops_per_image': round(GFLOPS,4), 'flops_method': flops_method,
    },
    'inference': {
        'bench_runs': BENCH_RUNS,
        'single_ms_mean': round(T_MEAN,4), 'single_ms_std': round(T_STD,4),
        'single_ms_p95': round(T_P95,4),   'single_fps': round(FPS_SINGLE,2),
        'batch_ms_mean': round(BT_MEAN,4), 'batch_per_image_ms': round(BT_PER_IMG,4),
        'batch_fps': round(FPS_BATCH,2),   'batch_size': BATCH_SIZE,
    },
    'gpu_memory': {
        'peak_train_mb': round(PEAK_TRAIN_MB,4), 'infer_alloc_mb': round(INFER_ALLOC_MB,4),
        'infer_cache_mb': round(INFER_CACHE_MB,4), 'infer_peak_mb': round(INFER_PEAK_MB,4),
    },
    'metrics': {k: (float(final[k]) if not isinstance(final[k], np.ndarray) else final[k].tolist())
                for k in ['loss','accuracy','emr','hamming','precision','recall',
                          'f1_micro','f1_macro','f1_weighted','auc_macro','auc_micro',
                          'map','mcc','f1_per','prec_per','rec_per','spec_per',
                          'auc_per','ap_per','mcc_per']},
    'history': history,
}

json_path = os.path.join(SAVE_DIR, 'results.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ JSON results saved → {json_path}")
print(f"✓ Best model: {best_path}")
print(f"✓ Curves: {os.path.join(SAVE_DIR, 'training_curves.png')}")
print(f"✓ Per-class: {os.path.join(SAVE_DIR, 'per_class_metrics.png')}")
print("\nSummary:")
print(f"  Experiment: {EXP_NAME}")
print(f"  Stage 4   : SpatialMambaLayer")
print(f"  Dataset   : Indiana University Chest X-ray")
print(f"  Val F1mic : {best_f1:.4f}")
print(f"  Model     : {total_params:,} params")
print(f"  GFLOPs    : {GFLOPS:.4f}")

✓ JSON results saved → /kaggle/working/checkpoints/spatial_mamba_indiana_20260503_213433/results.json
✓ Best model: /kaggle/working/checkpoints/spatial_mamba_indiana_20260503_213433/best_model.pth
✓ Curves: /kaggle/working/checkpoints/spatial_mamba_indiana_20260503_213433/training_curves.png
✓ Per-class: /kaggle/working/checkpoints/spatial_mamba_indiana_20260503_213433/per_class_metrics.png

Summary:
  Experiment: spatial_mamba_indiana_20260503_213433
  Stage 4   : SpatialMambaLayer
  Dataset   : Indiana University Chest X-ray
  Val F1mic : 0.4064
  Model     : 20,052,302 params
  GFLOPs    : 3.8323


## ✓ Experiment Complete

**Output files in `/kaggle/working/checkpoints/spatial_mamba_indiana_.../`:**
- `best_model.pth`
- `results.json` ← compare with baseline `results.json`
- `report.txt`
- `training_curves.png`
- `per_class_metrics.png`
- `training_log.txt`

**Compare with baseline:**
```python
import json
base = json.load(open('/kaggle/working/checkpoints/baseline_.../results.json'))
expr = json.load(open('/kaggle/working/checkpoints/spatial_mamba_.../results.json'))
for k in ['accuracy','f1_micro','f1_macro','auc_macro','map','mcc']:
    b = base['val_metrics']['overall'][k]
    e = expr['val_metrics']['overall'][k]
    print(f"{k:<20}: base={b:.4f}  expr={e:.4f}  delta={e-b:+.4f}")
```

**Download:** Kaggle → Output tab → Download all outputs